# THRESHOLD

In [8]:
import h5py
import numpy as np
import pickle

# Open the samples idxs above 5M
filename = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/refining/data_slash_results/archs4_gene_human_v2.1_normalized_LW20230126.h5"

with h5py.File(filename, 'r') as f:
    raw_matrix = f['data/raw_matrix'][:]
    all_biosample_names = f['meta/biosamples/all_biosample_names'][:]

def filter_biosamples(matrix, threshold):
    column_sums = np.sum(matrix, axis=0)
    selected_cols_idx = column_sums >= threshold
    subset = matrix[:, selected_cols]
    return subset, selected_cols

threshold = 5000000
threshold_matrix, selected_cols = filter_biosamples(raw_matrix, threshold)
threhold_biosample_names = all_biosample_names[selected_cols_idx]


In [9]:
print(raw_matrix.shape)
print(threshold_matrix.shape)
print(len(threhold_biosample_names))

(57447, 192601)
(57447, 87872)
87872


In [8]:
# Open the h5py file
with h5py.File(filename, 'a') as f:  # Get the matrix from the file

    # Save the new matrix in the h5py file
    del f["data/threshold_matrix"]
    f.create_dataset('data/threshold_matrix', data=threshold_matrix)
    
    del f["meta/biosamples/threshold_biosample_names"]
    f.create_dataset('meta/biosamples/threshold_biosample_names', data=threhold_biosample_names)

In [7]:
# TEST CODE

# raw_matrix = np.array([[1, 2, 3],
#                    [4, 5, 6],
#                    [7, 8000000, 9]])
# all_biosample_names = np.array(["b1", "b2", "b3"])

# def filter_biosamples(matrix, threshold):
#     column_sums = np.sum(matrix, axis=0)
#     selected_cols = column_sums >= threshold
#     subset = matrix[:, selected_cols]
#     return subset, selected_cols

# threshold = 5000000
# threshold_matrix, selected_cols = filter_biosamples(raw_matrix, threshold)
# selected_col_names = all_biosample_names[selected_cols]

# print(threshold_matrix)
# print()
# print(selected_col_names)

[[      2]
 [      5]
 [8000000]]

['b2']


# CORRECTED

In [15]:
import h5py
import csv
import numpy as np

# Open the CSV file
gene_length_list = []

# Open the CSV file
folder = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/refining/data_slash_results/"
with open(folder + 'edited_gene_name_length_idx_data.csv', 'r') as f:
    # Skip the header row
    next(f)

     # Read the remaining rows
    reader = csv.reader(f)
    for row in reader:
        gene_length_list.append(row[3])

gene_length_list = [float(x) for x in gene_length_list]
print("Finished reading gene length information.")

# Open the h5py file
with h5py.File(filename, 'a') as hf:
    # Get the matrix from the file
    matrix = hf['data/threshold_matrix']
    matrix = np.array(matrix)
    matrix = matrix.astype(np.float32)
    print("Finished reading matrix.")

    # Divide all the values in the matrix by the corresponding values in the gene_length_list
    for i, length in enumerate(gene_length_list):
        if length == 0:
            continue
        matrix[i, :] = matrix[i, :] / length
    print("Finished correcting matrix.")
     

    # Save the new matrix in the h5py file
    hf.create_dataset('data/corrected_matrix2', data=matrix)

    # Create a dataset for the gene_length_list
    hf.create_dataset('meta/genes/gene_length', data=gene_length_list)

    # Delete the original matrix from the file
    #del hf['matrix']

    print("Finished saving corrected matrix and gene length information.")


Finished reading gene length information.
Finished reading matrix.
Finished correcting matrix.
Finished saving corrected matrix and gene length information.


In [18]:
print(threshold_matrix[0][0:5])
print(matrix[0][0:5])
# First gene length: 3374

[  0   4   7 108  78]
[0.         0.00118554 0.00207469 0.03200949 0.02311796]


In [19]:
# Open the h5py file
with h5py.File(filename, 'a') as f:  # Get the matrix from the file

    # Save the new matrix in the h5py file
    del f["corrected_matrix2"]
    del f["gene_length2"]
    del f["data/corrected_matrix"]
    f.create_dataset('data/corrected_matrix', data=matrix)


# QN

In [33]:
import sys  
!{sys.executable} -m pip install --user qnorm

  Using cached qnorm-0.8.1-py3-none-any.whl (15 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 20.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 23.1 MB/s eta 0:00:0000:0100:01


In [34]:
#!conda install qnorm
import qnorm 

# Open the h5py file
with h5py.File(filename, 'r+') as hf:
    # Get the matrix from the file
    matrix = hf['data/corrected_matrix']

    # Quantile normalize 
    qnorm_matrix = qnorm.quantile_normalize(np.log2(1+np.array(matrix)))

    # Save the new matrix in the h5py file
    hf.create_dataset('data/qn_matrix', data=qnorm_matrix)


In [38]:
# Open the h5py file
with h5py.File(filename, 'a') as f:  # Get the matrix from the file

    # Save the new matrix in the h5py file
    del f["qn_matrix"]
    del f["data/qn_matrix"]
    f.create_dataset('data/qn_matrix', data=qnorm_matrix)
